# 학습 및 검증

### 1) 필요한 라이브러리 설치

In [ ]:
import pandas as pd
import os
import re
import json
import yaml
from datetime import datetime
from glob import glob
from tqdm import tqdm
from pprint import pprint
import torch
import pytorch_lightning as pl
from rouge import Rouge # 모델의 성능을 평가하기 위한 라이브러리입니다.

from torch.utils.data import Dataset , DataLoader
from transformers import AutoTokenizer, BartForConditionalGeneration, BartConfig
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
from transformers import AutoConfig, AutoModelForSeq2SeqLM, AutoTokenizer


import wandb # 모델 학습 과정을 손쉽게 Tracking하고, 시각화할 수 있는 라이브러리입니다.

### 2) Config file 만들기

In [4]:
# config 설정에 tokenizer 모듈이 사용되므로 미리 tokenizer를 정의해줍니다.
tokenizer = AutoTokenizer.from_pretrained("KETI-AIR/ke-t5-large", use_fast=False)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
config_data = {
    "general": {
        "data_path": "../../../data/", # 모델 생성에 필요한 데이터 경로를 사용자 환경에 맞게 지정합니다.
        "model_name": "KETI-AIR/ke-t5-large", # 불러올 모델의 이름을 사용자 환경에 맞게 지정할 수 있습니다.
        "output_dir": "./" # 모델의 최종 출력 값을 저장할 경로를 설정합니다.
    },
    "tokenizer": {
        "encoder_max_len": 512,
        "decoder_max_len": 128,
        # "bos_token": f"{tokenizer.bos_token}", # T5 모델은 bos_token이 따로 없으므로 주석 처리합니다.
        "eos_token": f"{tokenizer.eos_token}",
        # 특정 단어들이 분해되어 tokenization이 수행되지 않도록 special_tokens을 지정해줍니다.
        "special_tokens": ['#Person1#', '#Person2#', '#Person3#', 
                           '#Person4#', '#Person5#', '#Person6#', '#Person7#',
                           '#PhoneNumber#', '#Address#', '#PassportNumber#']
    },
    "training": {
        "overwrite_output_dir": True,
        "num_train_epochs": 20,
        "learning_rate": 1e-5,
        "max_grad_norm": 1.0,
        "per_device_train_batch_size": 2,
        "per_device_eval_batch_size": 2,
        "warmup_ratio": 0.1,
        "weight_decay": 0.01,
        "lr_scheduler_type": 'cosine',
        "optim": 'adamw_torch',
        "gradient_accumulation_steps": 32,
        "evaluation_strategy": 'epoch',
        "save_strategy": 'epoch',
        "save_total_limit": 5,
        "fp16": False,
        "load_best_model_at_end": True,
        "seed": 42,
        "logging_dir": "./logs",
        "logging_strategy": "steps",
        "predict_with_generate": True,
        "generation_max_length": 100,
        "do_train": True,
        "do_eval": True,
        "early_stopping_patience": 3,
        "early_stopping_threshold": 0.001,
        "report_to": "wandb", # (선택) wandb를 사용할 때 설정합니다.
        "dataloader_num_workers": 0,
        "dataloader_pin_memory": False,
        "logging_steps": 10,
        "gradient_checkpointing": True
    },
    # (선택) wandb 홈페이지에 가입하여 얻은 정보를 기반으로 작성합니다.
    "wandb": {
        "entity": "imeanseo_",
        "project": "dialogue-summarization",
        "name": "t5_large_baseline_v3_test"
    },
    "inference": {
        "ckt_path": "model ckt path", # 사전 학습이 진행된 모델의 checkpoint를 저장할 경로를 설정합니다.
        "result_path": "../../prediction/",
        "no_repeat_ngram_size": 2,
        "early_stopping": True,
        "generate_max_length": 100,
        "num_beams": 4,
        "batch_size" : 16,
        "remove_tokens": [f"{tokenizer.eos_token}", f"{tokenizer.pad_token}"]
    }
}

In [ ]:
# 모델의 구성 정보를 YAML 파일로 저장합니다.
config_path = "../../../config.yaml"
with open(config_path, "w") as file:
    yaml.dump(config_data, file, allow_unicode=True)

### 3) Configuration 불러오기

In [ ]:
# 저장된 config 파일을 불러옵니다.
with open(config_path, "r") as file:
    loaded_config = yaml.safe_load(file)

# 불러온 config 파일의 전체 내용을 확인합니다.
pprint(loaded_config)

{'general': {'data_path': '../data/',
             'model_name': 'KETI-AIR/ke-t5-large',
             'output_dir': './'},
 'inference': {'batch_size': 16,
               'ckt_path': 'model ckt path',
               'early_stopping': True,
               'generate_max_length': 100,
               'no_repeat_ngram_size': 2,
               'num_beams': 4,
               'remove_tokens': ['</s>', '<pad>'],
               'result_path': './prediction/'},
 'tokenizer': {'decoder_max_len': 128,
               'encoder_max_len': 512,
               'eos_token': '</s>',
               'special_tokens': ['#Person1#',
                                  '#Person2#',
                                  '#Person3#',
                                  '#Person4#',
                                  '#Person5#',
                                  '#Person6#',
                                  '#Person7#',
                                  '#PhoneNumber#',
                                  '#Address#',
    

### 4) 전처리 데이터 로드

In [ ]:
# 전처리 데이터 불러오기
data_dir = "../../../data/"
train_data_path = os.path.join(data_dir, "train_preprocessed.csv")
test_data_path = os.path.join(data_dir, "test_preprocessed.csv")
dev_data_path = os.path.join(data_dir, "dev_preprocessed.csv")

full_train_dataset = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
dev_data = pd.read_csv(dev_data_path)

print(f"✅ 전처리 된 학습 데이터 개수: {len(full_train_dataset)}")
print(f"✅ 전처리 된 테스트 데이터 개수: {len(test_data)}")
print(f"✅ 전처리 된 검증 데이터 개수: {len(dev_data)}")

✅ 전처리 된 학습 데이터 개수: 12457
✅ 전처리 된 테스트 데이터 개수: 499
✅ 전처리 된 검증 데이터 개수: 499


In [ ]:
# 미니 데이터셋 생성
# 전체 데이터셋에서 앞쪽 20개만 가져오기
train_data = full_train_dataset[:20] # HuggingFace Dataset인 경우

# 검증 데이터도 그냥 훈련 데이터랑 똑같은 걸로 넣어서 확인 (테스트용이니까)
dev_data = train_data

### 5) 데이터셋 클래스 구축

- 1. 토크나이징 & 데이터셋 클래스

In [9]:
import torch
from torch.utils.data import Dataset

class T5Dataset(Dataset):
    def __init__(self, df, tokenizer, config, is_train=True):
        self.tokenizer = tokenizer
        self.data = df
        self.is_train = is_train
        self.encoder_max_len = config['tokenizer']['encoder_max_len']
        self.decoder_max_len = config['tokenizer']['decoder_max_len']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # 1. 입력 데이터 (Dialogue) 토크나이징
        # 전처리된 dialogue 칼럼을 가져옵니다. (이미 "요약: " prefix가 붙어있어야 함)
        inputs = self.tokenizer(
            row['dialogue'],
            max_length=self.encoder_max_len,
            padding="max_length", # 혹은 'longest' (DataCollator 쓸거면 False 추천하지만 일단 안전하게)
            truncation=True,
            return_tensors="pt"
        )

        input_ids = inputs['input_ids'].squeeze(0)
        attention_mask = inputs['attention_mask'].squeeze(0)

        result = {
            'input_ids': input_ids,
            'attention_mask': attention_mask
        }

        # 2. 정답 데이터 (Summary) 토크나이징 - 학습/검증 때만 필요
        if self.is_train:
            targets = self.tokenizer(
                row['summary'],
                max_length=self.decoder_max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            
            labels = targets['input_ids'].squeeze(0)
            
            # [꿀팁] Padding 토큰(0)을 -100으로 치환 (Loss 계산에서 제외하기 위함)
            # T5는 패딩을 학습하면 안 됩니다!
            labels[labels == self.tokenizer.pad_token_id] = -100
            
            result['labels'] = labels

        # 추론(Inference) 단계에서는 ID가 필요할 수 있음
        if 'fname' in row:
            result['fname'] = row['fname']

        return result

- 2. 데이터셋 생성 함수

In [10]:
def prepare_t5_train_dataset(config, train_df, val_df, tokenizer):
    print('-'*10, 'T5 Dataset 생성 시작', '-'*10)
    
    # Train 데이터셋
    train_dataset = T5Dataset(
        train_df, 
        tokenizer, 
        config, 
        is_train=True
    )
    
    # Validation 데이터셋
    val_dataset = T5Dataset(
        val_df, 
        tokenizer, 
        config, 
        is_train=True
    )
    
    print(f"✅ Train 개수: {len(train_dataset)}")
    print(f"✅ Val 개수: {len(val_dataset)}")
    
    return train_dataset, val_dataset

### 6) 평가 지표 계산 함수 생성

In [11]:
import numpy as np
from rouge import Rouge

def compute_metrics(config, tokenizer, eval_pred):
    predictions, labels = eval_pred
    vocab_size = len(tokenizer)
    
    # 1. -100 (Ignore Index) 처리
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # 2. [중요] 0보다 작거나, 단어장 크기보다 큰 '미친 값'들은 전부 Pad로 변경
    # (IndexError와 OverflowError를 동시에 잡는 2중 필터)
    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)
    predictions = np.where(predictions >= vocab_size, tokenizer.pad_token_id, predictions)

    # 3. [핵심 해결책] Numpy 타입을 Python int로 강제 변환 (.astype(int))
    # OverflowError는 Numpy 자료형이 C++ 토크나이저와 충돌할 때 발생합니다.
    # 이를 막기 위해 순수 int로 변환합니다.
    decoded_preds = tokenizer.batch_decode(predictions.astype(int), skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels.astype(int), skip_special_tokens=True)

    decoded_preds = ["\n".join(pred.strip().split()) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split()) for label in decoded_labels]
    
    # 4. ROUGE 점수 계산
    rouge = Rouge()
    try:
        if len(decoded_preds) == 0 or len(decoded_labels) == 0:
             result = {"rouge-1": 0.0, "rouge-2": 0.0, "rouge-l": 0.0}
        else:
            results = rouge.get_scores(decoded_preds, decoded_labels, avg=True)
            result = {key: value["f"] for key, value in results.items()}
    except Exception as e:
        print(f"Rouge Error: {e}")
        result = {"rouge-1": 0.0, "rouge-2": 0.0, "rouge-l": 0.0}

    # gen_len 계산
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return result

### 7) 모델, 토크나이저, 트레이너 로드

In [ ]:

def load_tokenizer_and_model_for_train(config, device):
    print('-'*10, 'Load tokenizer & model', '-'*10)
    model_name = config['general']['model_name']
    print(f'Model Name : {model_name}')
    
    # 1. Config 로드
    model_config = AutoConfig.from_pretrained(model_name)
    
    # 2. Tokenizer 로드 (Fast 버전 에러나면 use_fast=False)
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    # 2-1. Special Tokens 추가
    special_tokens_dict = {'additional_special_tokens': config['tokenizer']['special_tokens']}
    num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
    print(f"✅ Added {num_added_toks} special tokens: {config['tokenizer']['special_tokens']}")
    
    # 3. Model 로드 (T5는 AutoModelForSeq2SeqLM 사용)
    generate_model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name, 
        config=model_config
    )

    generate_model.resize_token_embeddings(len(tokenizer))
    print(f"✅ Resized token embeddings to {len(tokenizer)}")
    
    # GPU로 이동
    generate_model.to(device)
    
    print('-'*10, 'Load tokenizer & model complete', '-'*10)
    return generate_model, tokenizer

In [13]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from transformers import EarlyStoppingCallback
import os

def load_trainer_for_train(config, generate_model, tokenizer, train_dataset, val_dataset):
    print('-'*10, 'Make training arguments', '-'*10)
    
    # 1. 학습 설정 (Hyperparameters)
    # config.yaml에 있는 값들을 가져와서 세팅합니다.
    training_args = Seq2SeqTrainingArguments(
        output_dir=config['general']['output_dir'],
        overwrite_output_dir=config['training']['overwrite_output_dir'],
        
        gradient_checkpointing=config['training']['gradient_checkpointing'],
        
        do_train=config['training']['do_train'],
        do_eval=config['training']['do_eval'],
        
        num_train_epochs=config['training']['num_train_epochs'],
        learning_rate=config['training']['learning_rate'],
        
        per_device_train_batch_size=config['training']['per_device_train_batch_size'],
        per_device_eval_batch_size=config['training']['per_device_eval_batch_size'],
        gradient_accumulation_steps=config['training']['gradient_accumulation_steps'],
        
        warmup_ratio=config['training']['warmup_ratio'],
        weight_decay=config['training']['weight_decay'],
        lr_scheduler_type=config['training']['lr_scheduler_type'],
        optim=config['training']['optim'],
        
        evaluation_strategy=config['training']['evaluation_strategy'],
        save_strategy=config['training']['save_strategy'],
        save_total_limit=config['training']['save_total_limit'],
        load_best_model_at_end=config['training']['load_best_model_at_end'],
        
        fp16=config['training']['fp16'],
        seed=config['training']['seed'],
        
        logging_dir=config['training']['logging_dir'],
        logging_strategy=config['training']['logging_strategy'],
        logging_steps=config['training']['logging_steps'],
        
        predict_with_generate=config['training']['predict_with_generate'],
        generation_max_length=config['training']['generation_max_length'],
        
        report_to=config['training']['report_to'],
        dataloader_num_workers=config['training']['dataloader_num_workers'],
    )

    # 2. [T5 필수] Data Collator 설정
    # 배치 내에서 가장 긴 문장에 맞춰서 동적으로 패딩을 해줍니다.
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=generate_model,
        label_pad_token_id=-100, # 정답 패딩 무시
        pad_to_multiple_of=8 if config['training']['fp16'] else None
    )

    # 3. Early Stopping (조기 종료)
    # 점수가 안 오르면 학습을 일찍 끝내서 과적합 방지
    my_callback = EarlyStoppingCallback(
        early_stopping_patience=config['training']['early_stopping_patience'],
        early_stopping_threshold=config['training']['early_stopping_threshold']
    )

    # 4. Trainer 생성
    trainer = Seq2SeqTrainer(
        model=generate_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator, # <--- 이거 꼭 있어야 함!
        compute_metrics=lambda pred: compute_metrics(config, tokenizer, pred),
        callbacks=[my_callback]
    )
    
    print('-'*10, 'Make trainer complete', '-'*10)
    return trainer

### 8) 학습 수행

In [ ]:
# checkpoint 디렉토리 설정을 위해 현재 시간을 기반으로 run_name을 자동 생성
now = datetime.now().strftime("%m%d_%H%M")
base_name = loaded_config['wandb']['name']
run_name = f"{base_name}_{now}"
loaded_config['general']['output_dir'] = f"../../../checkpoint/{run_name}/"

loaded_config['wandb']['name'] = run_name

print(f"🔥 Run Name 자동 생성: {run_name}")

🔥 Run Name 자동 생성: t5_large_baseline_v3_1201_2128


In [ ]:
def main(config):
    # 1. GPU 장치 설정
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print(f'Device : {device}')

    # 2. 모델 & 토크나이저 로드
    generate_model, tokenizer = load_tokenizer_and_model_for_train(config, device)

    # 3. 데이터 로드 및 전처리 (EDA 반영된 전처리 적용!)
    # (아까 만든 prepare_t5_train_dataset 함수 사용)
    # train_df, val_df 불러오는 코드는 위에서 이미 실행되어 있다고 가정
    train_dataset, val_dataset = prepare_t5_train_dataset(config, train_data, dev_data, tokenizer)

    # 4. Trainer 준비
    trainer = load_trainer_for_train(config, generate_model, tokenizer, train_dataset, val_dataset)

    # 5. WandB 초기화 (이름 덮어쓰기)
    # (config 파일 수정 안 하고 여기서 강제 주입)
    import wandb
    try:
        wandb.finish()
    except:
        pass
        
    wandb.init(
        entity=config['wandb']['entity'],
        project=config['wandb']['project'],
        name=config['wandb']['name'],
        config=config # 현재 설정 박제
    )

    # 6. 학습 시작! 🔥
    print("🔥 학습을 시작합니다! (T5-Large)")
    trainer.train()
    
    # 7. 종료
    wandb.finish()

# ==========================================
# 실행 트리거
# ==========================================
if __name__ == "__main__":
    # 메인 실행
    main(loaded_config)

### (서버 죽었을 때) checkpoint 찾아와서 재개

In [ ]:
# # 서버 뒤졌을때 심폐소생술

# import os
# from glob import glob
# from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# # 1. 저장된 폴더에서 가장 최신 체크포인트 자동 검색
# output_dir = config_data['general']['output_dir'] 
# checkpoints = glob(os.path.join(output_dir, "checkpoint-*"))

# if len(checkpoints) > 0:
#     # 숫자 기준으로 정렬해서 제일 큰 거 선택
#     latest_ckpt = sorted(checkpoints, key=lambda x: int(x.split('-')[-1]))[-1]
#     print(f"♻️ 발견된 최신 세이브 파일: {latest_ckpt}")
#     print("   -> 여기서부터 이어달리기 합니다! 🏃")
# else:
#     print("🚨 체크포인트가 없습니다! 처음부터 해야 합니다.")
#     latest_ckpt = None

# # ---------------------------------------------------------
# # 2. [핵심] 설정 강제 변경 (Batch 2 + Gradient Checkpointing)
# # 아까 4로 돌리다 죽었으니, 2로 줄여서 안전하게 갑니다.
# # ---------------------------------------------------------
# training_args = Seq2SeqTrainingArguments(
#     output_dir=output_dir,
#     overwrite_output_dir=False, # 덮어쓰기 금지! (이어해야 하니까)
    
#     # 🔥 [심폐소생술 설정] 절대 안 죽게 세팅
#     per_device_train_batch_size=2,        # 4 -> 2 (다이어트)
#     per_device_eval_batch_size=2,         # 4 -> 2
#     gradient_accumulation_steps=32,       # 16 -> 32 (배치 줄인 만큼 누적 늘리기)
#     gradient_checkpointing=True,          # ⭐ 필살기: 메모리 절약 모드 켜기
    
#     # 나머지 설정 유지
#     num_train_epochs=config_data['training']['num_train_epochs'],
#     learning_rate=config_data['training']['learning_rate'],
#     save_strategy='epoch',
#     save_total_limit=5,
#     fp16=False,
#     logging_steps=10,
#     predict_with_generate=True,
#     do_train=True,
#     do_eval=True,
#     report_to="wandb"
# )

# # 3. 모델 & 토크나이저 & 데이터셋 로드 (기존 함수 활용)
# device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# generate_model, tokenizer = load_tokenizer_and_model_for_train(config_data, device)
# train_dataset, val_dataset = prepare_t5_train_dataset(config_data, train_data, dev_data, tokenizer)

# # 4. Data Collator (T5 필수)
# data_collator = DataCollatorForSeq2Seq(
#     tokenizer=tokenizer,
#     model=generate_model,
#     label_pad_token_id=-100,
#     pad_to_multiple_of=8
# )

# # 5. Trainer 다시 만들기 (새 설정 적용)
# trainer = Seq2SeqTrainer(
#     model=generate_model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     data_collator=data_collator,
#     tokenizer=tokenizer,
#     compute_metrics=lambda pred: compute_metrics(config_data, tokenizer, pred)
# )

# # 6. WandB 재연결 (이어쓰기)
# import wandb
# try: wandb.finish() 
# except: pass

# wandb.init(
#     entity="imeanseo_",
#     project="dialogue-summarization",
#     name="t5_large_resume", # 이름 살짝 변경
#     resume=True             # 이어쓰기 모드
# )

# # 7. 🔥 이어하기 실행!
# if latest_ckpt:
#     print(f"🚀 {latest_ckpt} 부터 학습을 재개합니다...")
#     trainer.train(resume_from_checkpoint=latest_ckpt)
# else:
#     print("🚀 처음부터 다시 시작합니다 (안전 모드 적용)")
#     trainer.train()